# 01 — 因子研究（API-first）

**前置：** 已完成 `00_data_download_and_sync.ipynb`，或在本 notebook 中使用默认配置。

**职责：**
- 读取统一 `session_config.json`
- 使用 canonical raw 10D return：`Ref($close, -10) / $close - 1`
- 调用 `src.research.notebook_research_api.daily_correlation_table`
- 输出 `data/factor_selection.json` 供训练 notebook 读取

本 notebook 只保留配置、Qlib 数据加载和可视化；可复用研究逻辑已经下沉到 `src.research.*`。


In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common.qlib_init import build_qlib_init_cfg, safe_qlib_init
from src.research.notebook_lab_contracts import ResearchSessionConfig, CANONICAL_10D_RETURN_EXPR
from src.research.notebook_research_api import (
    build_factor_selection,
    daily_correlation_table,
    sanitize_factor_name,
    save_factor_selection,
)

cfg_path = ROOT / "data" / "session_config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    MARKET, SYMBOLS, BENCHMARK = cfg["market"], cfg["symbols"], cfg["benchmark"]
    TRAIN_START, TRAIN_END = cfg["train_start"], cfg["train_end"]
    TEST_START, TEST_END = cfg["test_start"], cfg["test_end"]
    print(f"Loaded session_config.json ({cfg.get('created_at', '')[:19]})")
else:
    MARKET = "us"
    SYMBOLS = ["AAPL", "NVDA", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "AVGO", "COST", "NFLX"]
    BENCHMARK = "QQQ"
    TRAIN_START, TRAIN_END = "2021-01-01", "2024-12-31"
    TEST_START, TEST_END = "2025-01-01", "2026-06-18"
    print("session_config.json not found; using notebook defaults")

config = ResearchSessionConfig(
    market=MARKET,
    symbols=SYMBOLS,
    benchmark=BENCHMARK,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    test_start=TEST_START,
    test_end=TEST_END,
    return_expression=CANONICAL_10D_RETURN_EXPR,
)

safe_qlib_init(build_qlib_init_cfg(None, market=MARKET))
from qlib.data import D

print(f"Market={MARKET.upper()}  Symbols={len(SYMBOLS)}  Test={TEST_START}->{TEST_END}")


In [ ]:
raw = D.features(
    SYMBOLS,
    [config.return_expression],
    start_time=TEST_START,
    end_time=TEST_END,
)
raw_returns = raw.copy()
raw_returns.columns = ["return"]
if raw_returns.index.names == ["instrument", "datetime"]:
    raw_returns = raw_returns.swaplevel().sort_index()
raw_returns.attrs["provenance"] = "raw_forward_return"
raw_returns.attrs["horizon"] = 10
raw_returns.attrs["expression"] = config.return_expression

print(f"Raw 10D returns: {raw_returns.shape}")
print(raw_returns.head())


In [ ]:
FACTOR_GROUPS = {
    "price_momentum": [
        "$close/Ref($close,5)-1",
        "$close/Ref($close,10)-1",
        "$close/Ref($close,20)-1",
        "$close/Ref($close,60)-1",
    ],
    "volatility": [
        "Std($close,10)",
        "Std($close,20)",
        "Std($ret,10)",
    ],
    "volume": [
        "$volume/Ref($volume,10)-1",
        "$volume/Mean($volume,20)-1",
    ],
    "alpha158_subset": [
        "Ref($close,1)/Ref($open,1)-1",
        "$high/$low-1",
        "Mean($close,5)/Mean($close,20)-1",
    ],
}
all_factors = [factor for group in FACTOR_GROUPS.values() for factor in group]
print(f"Total factors to research: {len(all_factors)}")

X_factors = D.features(SYMBOLS, all_factors, start_time=TEST_START, end_time=TEST_END)
if X_factors.index.names == ["instrument", "datetime"]:
    X_factors = X_factors.swaplevel().sort_index()
X_factors = X_factors.fillna(0.0).replace([np.inf, -np.inf], 0.0)
X_factors.columns = [sanitize_factor_name(factor) for factor in all_factors]

print(f"Factor matrix: {X_factors.shape}")


In [ ]:
ic_table = daily_correlation_table(X_factors, raw_returns, min_items_per_day=3)
display_table = ic_table.rename(columns={"name": "factor", "mean": "ic_mean", "ir": "ic_ir"})
print("Top factors by ICIR:")
display(display_table.head(10))

selection = build_factor_selection(ic_table, market=MARKET, label="raw_10d_return")
selection_path = save_factor_selection(selection, ROOT / "data" / "factor_selection.json")

print(f"Saved factor selection -> {selection_path}")
print(f"Good factors ({len(selection['good_factors'])}): {selection['good_factors']}")
print(f"Weak factors ({len(selection['weak_factors'])}): {selection['weak_factors']}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_df = display_table.sort_values("ic_mean")
axes[0].barh(range(len(plot_df)), plot_df["ic_mean"])
axes[0].set_yticks(range(len(plot_df)))
axes[0].set_yticklabels(plot_df["factor"], fontsize=7)
axes[0].axvline(0, linewidth=0.5)
axes[0].set_title("Single-factor IC Mean")

plot_ir = display_table.sort_values("ic_ir")
axes[1].barh(range(len(plot_ir)), plot_ir["ic_ir"])
axes[1].set_yticks(range(len(plot_ir)))
axes[1].set_yticklabels(plot_ir["factor"], fontsize=7)
axes[1].axvline(0, linewidth=0.5)
axes[1].set_title("Single-factor ICIR")

plt.tight_layout()
plt.show()


In [ ]:
corr = X_factors.corr(method="spearman").fillna(0.0)

high_corr = []
for i in range(len(corr)):
    for j in range(i + 1, len(corr.columns)):
        value = corr.iloc[i, j]
        if abs(value) > 0.8:
            high_corr.append((corr.index[i], corr.columns[j], value))

if high_corr:
    print("High-correlation pairs (|r|>0.8):")
    for left, right, value in sorted(high_corr, key=lambda item: -abs(item[2]))[:20]:
        print(f"  {left} × {right}: r={value:.3f}")
else:
    print("No high-correlation pairs above 0.8")
